In [1]:
%load_ext autoreload

In [2]:
%autoreload 2
import os
from pathlib import Path
import pandas as pd
from itertools import product
import shutil

from darpinstances.instance import load_instance_config
from darpinstances.instance_generation.generate_config import generate_config
from darpbenchmark.sizing import calculate_sizing_for_instance

PATH = Path.cwd()
INSTANCE_PATH_OLD = PATH.parents[2] / "Instances"
INSTANCE_PATH_RCI = PATH.parents[2] / "rci" / "Instances"
RESULTS_PATH = PATH.parents[2] / "sizing" / "Results"
RESULTS_PATH_RCI = PATH.parents[2] / "rci" / "Results"

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance.py:23: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
def create_custom_results_config(instance_path, method, outdir):
    config = {}
    config['instance'] = instance_path
    config['method'] = method
    config['outdir'] = outdir
    return config

In [18]:
# Chicago not for sizing (can't load DM - to)
cities = ['Sydney', 'DC', 'Manhattan', 'NYC']
# cities = ['Porto', 'Sydney', 'DC', 'Manhattan', 'Chicago', 'NYC']
# cities = ['NYC', 'DC', 'Chicago', 'Manhattan']
# cities = ['Porto', 'Sydney']
# starts_str = {'18-00': ['05_min']}
starts_str = {'18-00': ['05_min', '15_min', '30_min', '2_h']}
# starts = {18: [5]}
starts = {18: [5, 15, 30, 120]}
# start_durations = {7: [16*60], 18: [180, 1, 30, 15, 5]}
delays_str = ['03_min', '05_min', '10_min', '15_min']
# delays = [3]
delays = [3, 5, 10, 15]
methods = ['ih', 'vga', 'halns', 'vga_chaining']
# capacities = [4]
capacities = [4, 6, 10]

file_copy = 'requests.csv'
dir_copy = 'shapefiles'

rci = True
instance_setup = True
sizing = True

In [19]:
# generate shapefiles:
from darpinstances.instance_generation.map import NearestNodeProvider, get_map
from darpinstances.instance_generation.demand_generation import generate_demand
from darpinstances.instance_generation.vehicles import generate_vehicles

def generate_vehicle_shapefiles(city, instance_config, instance_dir_old, instance_dir_new):
    instance_config['area_dir'] = "../../../../"
    os.chdir(instance_dir_old)
    map_nodes, _ = get_map(instance_config)
    crs_metric = instance_config['map']['SRID_plane']
    nodes = map_nodes.to_crs(f'epsg:{crs_metric}')
    nearest_node_provider = NearestNodeProvider(nodes)

    requests = generate_demand(map_nodes, instance_config, nearest_node_provider, crs_metric)
    desired_vehicle_count = int(len(requests) * 1)

    os.chdir(instance_dir_new)
    instance_config['vehicles']['positions'] = 'random'
    generate_vehicles(map_nodes, instance_config, nearest_node_provider, desired_vehicle_count)
    os.chdir(PATH)


In [6]:
def modify_configs(instance_config, capacity, delay):
    instance_config['vehicles']['vehicle_capacity'] = capacity
    instance_config['demand']['filepath'] = 'requests.csv'
    instance_config['max_prolongation'] = delay*60
    instance_config['area_dir'] = "../../../../../"
    instance_config['vehicles'].pop('vehicle_count', None)
    instance_config['map'].pop('path', None)
    instance_config.pop('instance_dir', None)
    return instance_config

In [14]:
def setup_config_rci(vehicle_count, city, start_str, i, j, capacity, delay):
    # setup new instance config
    instance_path_relative = Path(f'{city}/instances/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}')
    instance_dir_new = INSTANCE_PATH_RCI / instance_path_relative
    instance_path = instance_dir_new / "config.yaml"
    os.makedirs(instance_dir_new, exist_ok=True)

    # load old config
    instance_dir_old = INSTANCE_PATH_OLD / instance_path_relative
    instance_config_path = instance_dir_old / 'config.yaml'

    inst_conf_dict = load_instance_config(instance_config_path)
    instance_config = modify_configs(inst_conf_dict, capacity, delay)
    if vehicle_count:
        instance_config['vehicles']['vehicle_count'] = vehicle_count

    # generate new config
    generate_config(instance_config, instance_path)

    # copy requests and vehicles
    files = ['requests.csv', 'vehicles.csv']
    for f in files:
        src = instance_dir_old / f
        dst = instance_dir_new / f
        if src.exists():
            shutil.copy(src, dst)

    # setup new results configs
    for method in methods:

        results_dir = RESULTS_PATH_RCI / f'{city}/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/{method}'
        os.makedirs(results_dir, exist_ok=True)
        results_path = results_dir / "config.yaml"

        if rci:
            instance_path =  'Instances' / instance_path_relative / 'config.yaml'
            results_dir = f'Results/{city}/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/{method}'
        
        results_config = create_custom_results_config(str(instance_path), method, str(results_dir))

        generate_config(results_config, results_path)

In [20]:
for start, durations in starts.items():
    start_str = f"{start:02d}-00"

    for (i, duration), (j, delay), city, capacity in product(
        enumerate(durations), enumerate(delays), cities, capacities
    ):
        
        # setup new instance config
        instance_path_relative = Path(f'{city}/instances/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}')
        instance_dir_new = INSTANCE_PATH_OLD / instance_path_relative
        instance_path = instance_dir_new / "config.yaml"

        # load old config
        instance_dir_old = INSTANCE_PATH_OLD / f'{city}/instances/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[0]}'
        instance_config_path = instance_dir_old / 'config.yaml'
        if not instance_config_path.exists() and starts_str[start_str][i] == '2_h':
            instance_dir_old = INSTANCE_PATH_OLD / f'{city}/instances/start_{start_str}/duration_02_h/max_delay_{delays_str[0]}'
            instance_config_path = instance_dir_old / 'config.yaml'
            
        # modify config values
        inst_conf_dict = load_instance_config(instance_config_path)
        instance_config = modify_configs(inst_conf_dict, capacity, delay)
        
        if instance_setup:
            os.makedirs(instance_dir_new, exist_ok=True)

            # generate new config
            generate_config(instance_config, instance_path)

            # copy requests
            f = 'requests.csv'
            src = instance_dir_old / f
            dst = instance_dir_new / f
            if src.exists():
                req_df = pd.read_csv(src, delimiter='\t')
                if len(req_df.columns) == 4:
                    req_df = req_df.iloc[:, :-1]
                req_df.to_csv(dst, index=False, sep='\t')


            # copy shapefile directory
            src = instance_dir_old / dir_copy
            dst = instance_dir_new / dir_copy
            if src.exists():
                shutil.copytree(src, dst, dirs_exist_ok=True)

            generate_vehicle_shapefiles(city, instance_config, instance_dir_old, instance_dir_new)
            

        # setup new results configs for sizing (IH)
        if sizing:
            method = 'ih'
            results_dir = RESULTS_PATH / f'{city}/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/{method}'
            os.makedirs(results_dir, exist_ok=True)
            results_path = results_dir / "config.yaml"

            results_config = create_custom_results_config(str(instance_path), method, str(results_dir))

            generate_config(results_config, results_path)
            vehicle_count = calculate_sizing_for_instance(results_path)
        else:
            vehicle_count = None

        # setup new instance configs for RCI
        setup_config_rci(vehicle_count, city, start_str, i, j, capacity, delay)
        

13:07:03 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_05_min/max_delay_03_min/config.yaml
13:07:03 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_4/config.yaml
13:07:03 [INFO] Loading nodes from /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/map/nodes.csv
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance_generation/map.py:131: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  nodes = pd.read_csv(nodes_file_path, index_col=None, delim_whitespace=True)
13:07:03 [INFO] Loading edges from /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/map/edges.csv
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance_generation/map.py:13

/home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_4/sizing.csv


13:07:04 [INFO] Loading edges from /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/map/edges.csv
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance_generation/map.py:139: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  edges = pd.read_csv(edges_file_path, index_col=None, delim_whitespace=True)
13:07:05 [INFO] The demand file is already in /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_05_min/max_delay_03_min/requests.csv, skipping demand generation.
13:07:05 [INFO] Saving vehicles to /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_6/vehicles.csv
/usr/lib/python3/dist-packages/pyproj/crs/crs.py:131: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When mak

/home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_6/sizing.csv


/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance_generation/map.py:139: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  edges = pd.read_csv(edges_file_path, index_col=None, delim_whitespace=True)
13:07:05 [INFO] The demand file is already in /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_05_min/max_delay_03_min/requests.csv, skipping demand generation.
13:07:05 [INFO] Saving vehicles to /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_10/vehicles.csv
/usr/lib/python3/dist-packages/pyproj/crs/crs.py:131: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#a

...

/home/dominika/Desktop/deathOFbachelor/Instances/Manhattan/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_6/sizing.csv


13:07:08 [INFO] The demand file is already in /home/dominika/Desktop/deathOFbachelor/Instances/Manhattan/instances/start_18-00/duration_05_min/max_delay_03_min/requests.csv, skipping demand generation.
13:07:08 [INFO] Saving vehicles to /home/dominika/Desktop/deathOFbachelor/Instances/Manhattan/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_10/vehicles.csv
/usr/lib/python3/dist-packages/pyproj/crs/crs.py:131: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
13:07:08 [INFO] Saving shapefile with vehicles to: /home/dominika/Desktop/deathOFbachelor/Instances/Manhattan/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_10/shapefiles/vehicles.shp
13:07:08 [INFO] Saving config to /home/domini

/home/dominika/Desktop/deathOFbachelor/Instances/Manhattan/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_10/sizing.csv


13:07:09 [INFO] Loading edges from /home/dominika/Desktop/deathOFbachelor/Instances/NYC/map/edges.csv
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance_generation/map.py:139: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  edges = pd.read_csv(edges_file_path, index_col=None, delim_whitespace=True)
13:07:09 [INFO] The demand file is already in /home/dominika/Desktop/deathOFbachelor/Instances/NYC/instances/start_18-00/duration_05_min/max_delay_03_min/requests.csv, skipping demand generation.
13:07:09 [INFO] Saving vehicles to /home/dominika/Desktop/deathOFbachelor/Instances/NYC/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_4/vehicles.csv
/usr/lib/python3/dist-packages/pyproj/crs/crs.py:131: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the c

KeyboardInterrupt: 

In [ ]:
from itertools import product
import shutil
ic = 0
rc = 0
for start, durations in starts.items():
    start_str = f"{start:02d}-00"

    for (i, duration), (j, delay), city, capacity in product(
        enumerate(durations), enumerate(delays), cities, capacities
    ):
        
        # setup new instance config
        instance_path_relative = Path(f'{city}/instances/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}')
        instance_dir_new = INSTANCE_PATH / instance_path_relative
        instance_path = instance_dir_new / "config.yaml"

        # load old config
        instance_dir_old = INSTANCE_PATH_OLD / f'{city}/instances/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[0]}'
        instance_config_path = instance_dir_old / 'config.yaml'
        if not instance_config_path.exists() and starts_str[start_str][i] == '2_h':
            instance_dir_old = INSTANCE_PATH_OLD / f'{city}/instances/start_{start_str}/duration_02_h/max_delay_{delays_str[0]}'
            instance_config_path = instance_dir_old / 'config.yaml'
            
        # modify config values
        instance_config = modify_configs(load_instance_config(instance_config_path), capacity, delay)
        
        if instance_setup:
            os.makedirs(instance_dir_new, exist_ok=True)

            # generate new config
            generate_config(instance_config, instance_path)

            # copy requests
            src = instance_dir_old / file_copy
            dst = instance_dir_new / file_copy
            if src.exists():
                shutil.copy(src, dst)

            # copy shapefile directory
            src = instance_dir_old / dir_copy
            dst = instance_dir_new / dir_copy
            if src.exists():
                shutil.copytree(src, dst, dirs_exist_ok=True)

            generate_vehicle_shapefiles(instance_config, instance_dir_old, instance_dir_new)
            
            ic += 1

        # setup new results configs
        for method in methods:

            results_dir = RESULTS_PATH / f'{city}/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/{method}'
            os.makedirs(results_dir, exist_ok=True)
            results_path = results_dir / "config.yaml"

            if rci:
                instance_path =  'Instances' / instance_path_relative / 'config.yaml'
                results_dir = f'/Results/{city}/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/{method}'
            
            results_config = create_custom_results_config(str(instance_path), method, str(results_dir))

            results_dir = RESULTS_PATH / f'{city}/start_{start_str}/duration_{starts_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/{method}'
            
            generate_config(results_config, results_path)
            rc += 1

  
# print(f'{ic} instance configs created')
# print(f'{rc} instance configs created')

10:33:25 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/NYC/instances/start_18-00/duration_05_min/max_delay_03_min/config.yaml
10:33:25 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/generated/Instances/NYC/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_4/config.yaml


10:33:25 [INFO] Loading nodes from /home/dominika/Desktop/deathOFbachelor/Instances/NYC/map/nodes.csv
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance_generation/map.py:132: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  nodes = pd.read_csv(nodes_file_path, index_col=None, delim_whitespace=True)
10:33:25 [INFO] Loading edges from /home/dominika/Desktop/deathOFbachelor/Instances/NYC/map/edges.csv
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance_generation/map.py:140: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  edges = pd.read_csv(edges_file_path, index_col=None, delim_whitespace=True)
10:33:26 [INFO] The demand file is already in /home/dominika/Desktop/deathOFbachelor/Instances/NYC/instances/start_18-

192 instance configs created
768 instance configs created


# TODO
- generate instances with new configs
- run sizing on these instances
- adjust configs for these instances based on sizing
- reproduce new instances for rci